<a href="https://colab.research.google.com/github/Luuuuuig/Bachelor-End-Project/blob/main/Focal_Case_Selection_2026-10-07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Selecting the focal improvement activity
**Report version, 12 October 2026 · evidence from the original observation codes · scores and weights provisional until Zhongxin's review**

## Executive summary

**What this notebook answers.** SQ4 of the [Plan of Work](https://github.com/Luuuuuig/Bachelor-End-Project/blob/main/docs/proposal/Plan_of_Work_Final_2026-09-22.pdf): which improvement activity should be selected, with Arno's workload as the primary objective. Six candidate areas from the [Arno analysis](https://github.com/Luuuuuig/Bachelor-End-Project/blob/main/Arno_Analysis_2026_10_02.ipynb) pass through six feasibility gates and are then compared on seven criteria with swing weights; sensitivity checks show which judgements could change the result.

**Provisional decision.** **Order and confirmation checking leads with 76.2 preference points**, followed by article/supplier data (56.8), maximalisatie (51.9) and request intake & validation (50.8). The decision is provisional: the weights are Yijie's of 7 October, the scores are Yijie's judgement, and Zhongxin has reviewed neither. It is final only when four conditions are met (Section 6): suitable documents and reference answers from the Exact test account; Arno's acceptance of a reading, matching and flagging aid, settled before his leave at the end of October; Zhongxin's review of criteria, anchors, scores and weights; Johan's confirmation of the business value.

**Where to read.** The rows follow the numbered questions of the [update of 15 October](https://github.com/Luuuuuig/Bachelor-End-Project/blob/main/docs/analyze/Supervisor_Update_2026-10-15.md); questions 1, 2, 4 and 7 are answered in the analysis notebook.

| Question or decision | Section | What you will find | Input needed from you |
|---|---|---|---|
| 3. The criteria and their anchors | 3 | Seven criteria with definitions, safeguards against double counting and the 1/3/5 anchors | Your review: are they distinct enough, are the anchors right |
| 3. The weights | 3 | Yijie's provisional swing weights with their rationale | Option A (your own swings, as an informed review) or option B (review of Yijie's) |
| 5. Arno: written confirmation or a session | 6, condition 2 | What must be settled with Arno: what a check compares, where the trusted price comes from, which support he accepts | Your view on a written confirmation instead of a full session |
| 6. The evaluation participant | 6, condition 2 | Arno is on leave from the end of October | Who takes part in the evaluation, and when |
| Targeted checks before the leave | 2, Appendix A | The open gates and the checks that settle them (PC1 to PC4 for paired documents and a reference price) | Which checks you want done first |
| The scores and their evidence | 4, Appendix B | The score matrix with low/high ranges, the evidence string behind every score, the ranking, and every score and range before and after the 12 October rebuild | Your review of the scores and ranges |
| How robust the ranking is | 5, Appendix C | Single-change and combined-change checks, the stress case and random draws | Which ranges or judgements you dispute |
| The candidate areas and the gates | 1, 2 | Six candidates on the raw codes with their minutes; six gates per candidate with verdicts and reasons | Whether a boundary or a gate verdict is too lenient or too strict |

**Main results**

- **Ranking:** checking 76.2, article/supplier data 56.8, maximalisatie 51.9, request intake 50.8; checking scores highest or joint highest on every criterion (4).
- **Robustness:** checking stays first in all 14 weight changes and all 56 one-point score changes (smallest lead 14.1); with its workload score at 3 it leads 65.4 to 56.8; request intake scored on the whole Specification stage reaches 57.8; six combined one-point changes within the ranges put article data ahead; in the joint stress case request intake leads at 73.5 (5).
- **Gates:** two candidates rejected (PO supplier communication at G6, standard / review / manual at G5); four scored as provisional scenarios with open data and reference gates (2).
- **Weights:** workload 27.0%, data readiness 16.2%, AI fit and evaluation feasibility 13.5% each; Yijie's provisional swings, not Zhongxin's (3).
- **Checking boundary:** the CHECK code alone, 21 rows and 91 minutes, 37 in Ordering and 54 in Monitoring; the notes name a price or date for 33 minutes, an unspecified mismatch for 20 (1).

**Scope.** The same five observation CSVs and the same preparation as the Arno analysis: 892 observed minutes, 742 recorded after the lost-focus exclusion of two 8 September records, 182 former EXC minutes labelled and outside the comparison, so **560 minutes** are the denominator. Candidates overlap and are not added up. The evidence comes from the original codes, the agreed stages and the recorded notes; the kinds-of-work labels are archived. The method (gates, criteria, anchors, weights, sensitivity) is the one of 7 October; in the 12 October rebuild no score changed and two uncertainty ranges of request intake widened.

**How to read each step.** Every section states its purpose and approach, shows the inputs and the calculation, and then gives the result and its interpretation.

<a id="decision"></a>
## 1. Candidate boundaries and evidence

**Purpose and approach.** Define each candidate as a set of in-scope records and compute its recorded workload. The definitions are those of part 8 of the Arno analysis: checking = all CHECK rows; request intake = all rows in the Specification stage; standard / review / manual = all PO rows; PO supplier communication = all SEND rows; maximalisatie and article/supplier data = the note-based ledgers, copied row by row. The records are prepared exactly as in the analysis notebook (duration, lost-focus exclusion, code families, former EXC labelled, agreed stage column); the loading cell checks the input version.

In [1]:
from pathlib import Path
import hashlib
import sys
import numpy as np
import pandas as pd
from IPython.display import display

# The same input files, read the same way as in the Arno analysis (part 1): five observation CSVs, local first
dates = ["2026-08-31", "2026-09-01", "2026-09-08", "2026-09-18", "2026-09-23"]
csv_files = [f"Arno_Measurement_{date}.csv" for date in dates]

data_folder = Path("docs/measurement")          # when run from the repository root ...
if not data_folder.is_dir():
    data_folder = Path(".")                     # ... otherwise next to the notebook

missing_files = [name for name in csv_files if not (data_folder / name).is_file()]
if missing_files and "google.colab" in sys.modules:
    from google.colab import files
    print("Upload the missing input files:", missing_files)
    uploaded = files.upload()
    for name in csv_files:
        if name in uploaded:
            (data_folder / name).write_bytes(uploaded[name])
for name in csv_files:
    if not (data_folder / name).is_file():
        raise FileNotFoundError(f"Missing {name}. Put the five observation CSVs in {data_folder} and rerun.")
print("Input folder:", data_folder.resolve())

# Input version: repository commit 00b9802 (12 October 2026); the hashes identify the exact CSVs
input_commit = "00b9802ba6335076e27d6b674ef8de38f3c03bd6"
expected_sha256 = {
    "2026-08-31": "7e44dd9a94b0bd530bd23eca50a87e03abd1c4f147734e3210b509a695c8d4f0",
    "2026-09-01": "b7cc7cc9bffa0e60ffbaeaeebc4e7018062a806dde0533787fde395f72675db5",
    "2026-09-08": "165e0d36e7795a5ae61eae96d530973d8c84a3e5b012bf215be6d656706772b4",
    "2026-09-18": "574491604e7ad1d2d190783609e14f9ce52caff67b674d05c452d5bd45db5860",
    "2026-09-23": "7a5b581d603e27a3cabe8612d082ec7f529c3128888d510f81e62b5f279da932",
}

# Read each day; keep every field as text, so blanks and markers such as "/" and "—" stay visible
frames = []
for date in dates:
    path = data_folder / f"Arno_Measurement_{date}.csv"
    if hashlib.sha256(path.read_bytes()).hexdigest() != expected_sha256[date]:
        print(f"WARNING: {path.name} differs from the file in commit {input_commit[:7]}")
    df = pd.read_csv(path, encoding="utf-8-sig", dtype=str, keep_default_na=False)
    df = df.rename(columns={"Activity as written": "Activity", "Result / short note": "Note"})
    df["date"] = pd.Timestamp(date)
    df["source_row"] = df.index + 1          # one-based row in the CSV, after the header
    frames.append(df)
observations = pd.concat(frames, ignore_index=True)
assert not observations.duplicated(["date", "source_row"]).any()
print("Observation rows:", len(observations))

Input folder: /home/user/Bachelor-End-Project/docs/measurement
Observation rows: 287


In [2]:
# Prepare the records as in the Arno analysis (part 2): duration, one exclusion, code family, former EXC, stage, case-day
start = pd.to_datetime(observations["Start"], format="%H:%M", errors="coerce")
end = pd.to_datetime(observations["End"], format="%H:%M", errors="coerce")
observations["minutes"] = (end - start).dt.total_seconds() / 60          # untimed tallies stay missing, not zero

lost_focus = observations["date"].eq("2026-09-08") & observations["Case"].eq("OBS-16")   # loss of observer focus
included = observations.loc[~lost_focus].copy()

included["activity_group"] = included["Activity"].str.upper().str.strip().str.split("-").str[0]
included["activity_group"] = included["activity_group"].replace({"[UNCERTAIN]": "UNCLEAR"})
included.loc[included["Activity"].str.upper().eq("REQ-EXACT"), "activity_group"] = "PO-REVISIT"   # an earlier PO (2 October)
included["former_exc"] = included["Activity"].str.upper().str.contains("EXC")                     # outside improvement scope (1 October)
included.loc[included["former_exc"], "activity_group"] = "OTHER (outside improvement)"
included["stage"] = included["Van Weele stage"].str.strip()      # the stage agreed on 12 October, read from the data
included["case_ref"] = included["date"].dt.strftime("%d %b") + " " + included["Case"]

in_scope = included.loc[~included["former_exc"]].copy()          # the records considered for improvement
scope_minutes = in_scope["minutes"].sum()

# Guards: the agreed frame
assert len(observations) == 287 and len(included) == 285
assert included["minutes"].sum() == 742 and scope_minutes == 560
assert included.loc[included["former_exc"], "minutes"].sum() == 182
assert not included["stage"].eq("").any()
print("Rows read:", len(observations), "- included:", len(included), "- recorded minutes:", int(included["minutes"].sum()),
      "- in scope:", int(scope_minutes))

Rows read: 287 - included: 285 - recorded minutes: 742 - in scope: 560


In [3]:
# Note-based source rows for two candidates, copied from the 12 October review (evidence: direct / mixed / contextual)
ledger_rows = []
def add_rows(candidate, date, rows, evidence, reason):
    for row in rows:
        ledger_rows.append((candidate, pd.Timestamp(date), row, evidence, reason))

MAX = "Maximalisatie"
add_rows(MAX, "2026-08-31", [37, 51], "direct", "Untimed DEC note names MAX; occurrence only")
add_rows(MAX, "2026-08-31", [38, 52], "mixed", "MAX and PO work share the recorded interval")
add_rows(MAX, "2026-09-01", [46, 47, 48, 49, 50], "mixed", "MAX observed inside a PO episode (MAXOBS=Y); search and outcome not separately timed")
add_rows(MAX, "2026-09-18", [58, 61, 64], "mixed", "Maximisatie, or its unsuccessful attempt, named within a PO interval")
add_rows(MAX, "2026-09-23", [57], "mixed", "Maximalisatie of an existing PO; internal timing not separated")
add_rows(MAX, "2026-08-31", [8, 10], "contextual", "Article added to an old PO; a MAX search is not recorded")
add_rows(MAX, "2026-09-08", [30, 31, 37, 40, 41, 42], "contextual", "Lines combined or added; the search itself is not recorded")
add_rows(MAX, "2026-09-18", [25, 30, 56], "contextual", "Lines added; no isolated consolidation time is measured")
add_rows(MAX, "2026-09-23", [2, 3, 4, 5, 8, 18, 39], "contextual", "Lines added to orders; the search or decision is not recorded")
add_rows(MAX, "2026-08-31", [39], "contextual", "Untimed return after a HOLD intention")
add_rows(MAX, "2026-09-01", [45], "contextual", "HOLD observed; MAX not captured (MAXOBS=?)")

ART = "Article/supplier data in Exact"
add_rows(ART, "2026-08-31", [19, 21, 23, 24], "direct", "Article input in Exact requested, entered, and information attached")
add_rows(ART, "2026-08-31", [25], "mixed", "Second item needs input; a question is asked")
add_rows(ART, "2026-09-01", [26, 33], "direct", "Request to add two items; article-master work continued the next day")
add_rows(ART, "2026-09-08", [26], "mixed", "New supplier found, added by a colleague, Exact updated; four minutes shared")
add_rows(ART, "2026-09-23", [53], "direct", "Request to create an item")
add_rows(ART, "2026-08-31", [20, 22, 26, 63], "contextual", "Same article-input case; fields and action split not recorded")
add_rows(ART, "2026-09-01", [27, 28], "contextual", "Routing and forwarding of the item request")
add_rows(ART, "2026-09-08", [25], "mixed", "Exact updated after a component case; the object is unspecified")
add_rows(ART, "2026-09-23", [54], "contextual", "Blank-note clarification after the item request")

ledger = pd.DataFrame(ledger_rows, columns=["candidate", "date", "source_row", "evidence", "reason"])
ledger_joined = ledger.merge(in_scope, on=["date", "source_row"], how="inner", validate="one_to_one")
print("Ledger rows:", len(ledger), "- joined to in-scope records:", len(ledger_joined))
ledger_joined.groupby(["candidate", "evidence"]).agg(rows=("Case", "size"), timed=("minutes", "count"), minutes=("minutes", "sum"))

Ledger rows: 50 - joined to in-scope records: 50


rows  timed  minutes
candidate                      evidence                        
Article/supplier data in Exact contextual     7      6     17.0
                               direct         7      4     16.0
                               mixed          3      3     10.0
Maximalisatie                  contextual    20     18     45.0
                               direct         2      0      0.0
                               mixed         11     11     28.0

In [4]:
# The candidate table on the raw definitions
candidate_rows = {
    "Order and confirmation checking": in_scope.loc[in_scope["activity_group"].eq("CHECK")],
    "Request intake & validation": in_scope.loc[in_scope["stage"].eq("Specification")],
    "Standard / review / manual": in_scope.loc[in_scope["activity_group"].eq("PO")],
    "PO supplier communication": in_scope.loc[in_scope["activity_group"].eq("SEND")],
    MAX: ledger_joined.loc[ledger_joined["candidate"].eq(MAX)],
    ART: ledger_joined.loc[ledger_joined["candidate"].eq(ART)],
}
definitions = {"Order and confirmation checking": "code CHECK", "Request intake & validation": "stage Specification",
               "Standard / review / manual": "code PO", "PO supplier communication": "code SEND",
               MAX: "note: MAX / lines added (ledger)", ART: "note: article or supplier data in Exact (ledger)"}
rows = []
for name, frame in candidate_rows.items():
    minutes = frame["minutes"].sum()
    by_stage = frame.groupby("stage")["minutes"].sum().sort_values(ascending=False)
    rows.append([name, definitions[name], len(frame), frame["minutes"].count(), minutes, minutes / 560 * 100,
                 frame["case_ref"].nunique(), frame["date"].nunique(),
                 frame.loc[frame["date"].eq("2026-09-18"), "minutes"].sum() / minutes * 100,
                 "; ".join(f"{stage} {m:.0f}" for stage, m in by_stage.loc[by_stage.gt(0)].items())])
candidates = pd.DataFrame(rows, columns=["candidate", "definition", "records", "timed", "minutes", "% of 560",
                                         "case_days", "days", "% on 18 Sep", "stage split (minutes)"]).set_index("candidate")
with pd.option_context("display.max_colwidth", 120):
    display(candidates.round(1))
print("Specification rows by code:", candidate_rows["Request intake & validation"].groupby("activity_group")["minutes"].agg(["size", "sum"]).to_dict("index"))

,definition,records,timed,minutes,% of 560,case_days,days,% on 18 Sep,stage split (minutes)
candidate,,,,,,,,,
Order and confirmation checking,code CHECK,21,21,91.0,16.2,19,5,62.6,Monitoring 54; Ordering 37
Request intake & validation,stage Specification,33,32,121.0,21.6,23,5,30.6,Specification 121
Standard / review / manual,code PO,63,62,165.0,29.5,50,5,17.6,Ordering 161; Follow-up and evaluation 4
PO supplier communication,code SEND,43,36,71.0,12.7,40,5,19.7,Ordering 49; Follow-up and evaluation 8; Specification 7; Monitoring 4; Not applicable 3
Maximalisatie,note: MAX / lines added (ledger),33,29,73.0,13.0,26,5,19.2,Ordering 73
Article/supplier data in Exact,note: article or supplier data in Exact (ledger),17,13,43.0,7.7,6,4,0.0,Ordering 27; Specification 12; Supplier selection 4


Specification rows by code: {'CLAR': {'size': 29, 'sum': 108.0}, 'OTHER': {'size': 1, 'sum': 6.0}, 'SEND': {'size': 3, 'sum': 7.0}}


In [5]:
# Candidate minutes by day, the dependence on 18 September, and the Specification rows whose note names a search
day_index = pd.to_datetime(dates)
candidate_by_day = pd.DataFrame({name: frame.groupby("date")["minutes"].sum().reindex(day_index, fill_value=0)
                                 for name, frame in candidate_rows.items()}).T
candidate_by_day.columns = day_index.strftime("%d %b")
scope_18_sep = in_scope.loc[in_scope["date"].eq("2026-09-18"), "minutes"].sum()
candidate_by_day["without 18 Sep"] = candidate_by_day.drop(columns="18 Sep").sum(axis=1)
candidate_by_day["without 18 Sep, % of 411"] = candidate_by_day["without 18 Sep"] / (scope_minutes - scope_18_sep) * 100
display(candidate_by_day.round(1))

# The gated request-intake scenario is finding old POs and drawings; the raw codes do not isolate that search,
# so the Specification rows whose note names a drawing, an old-PO lookup or an item history are listed
specification = candidate_rows["Request intake & validation"]
retrieval_like = specification.loc[specification["Note"].str.contains("drawing|lookup|history", case=False, regex=True)]
print("Specification rows whose note names a drawing, an old-PO lookup or an item history:", len(retrieval_like), "rows,",
      f"{retrieval_like['minutes'].sum():.0f} minutes ({retrieval_like['minutes'].sum() / scope_minutes * 100:.1f}% of 560),",
      retrieval_like["case_ref"].nunique(), "case-days on", retrieval_like["date"].nunique(), "days")
with pd.option_context("display.max_colwidth", 100):
    display(retrieval_like[["date", "source_row", "Case", "Activity", "minutes", "Note"]])

,31 Aug,01 Sep,08 Sep,18 Sep,23 Sep,without 18 Sep,"without 18 Sep, % of 411"
Order and confirmation checking,13.0,13.0,3.0,57.0,5.0,34.0,8.3
Request intake & validation,28.0,33.0,6.0,37.0,17.0,84.0,20.4
Standard / review / manual,22.0,51.0,25.0,29.0,38.0,136.0,33.1
PO supplier communication,15.0,18.0,14.0,14.0,10.0,57.0,13.9
Maximalisatie,10.0,15.0,10.0,14.0,24.0,59.0,14.4
Article/supplier data in Exact,17.0,11.0,8.0,0.0,7.0,43.0,10.5


Specification rows whose note names a drawing, an old-PO lookup or an item history: 6 rows, 36 minutes (6.4% of 560), 5 case-days on 3 days


,date,source_row,Case,Activity,minutes,Note
82,2026-09-01,20,OBS-12,CLAR,16.0,MI + historical search: old PO lookup; several matching items; required size/information not sup...
182,2026-09-18,28,OBS-10,CLAR,2.0,Look for the relevant drawings and ask the engineer for them.
185,2026-09-18,31,OBS-11,CLAR,4.0,Relevant person unavailable; ask other colleagues. Also discuss the OBS-10 drawing.
195,2026-09-18,41,OBS-10,CLAR,4.0,Final check that the drawing and its information were complete.
262,2026-09-23,34,OBS-12,CLAR,5.0,"Look for the relevant drawing; unable to find it, ask a colleague."
266,2026-09-23,38,OBS-14,CLAR,5.0,Look into the history of tubes following Emiel's request to order a tube.


**Result.** The largest areas are PO work (165 minutes, 29.5% of 560), the Specification stage as request intake (121 minutes, 21.6%, of which 108 are CLAR rows with a note) and order and confirmation checking (91 minutes, 16.2%). PO supplier communication has 71 minutes, maximalisatie 73 (mostly contextual: 45 minutes are rows where lines were added without a recorded MAX search, and the two direct MAX occurrences are untimed) and article/supplier data 43. Checking is the candidate that depends most on one day: 57 of its 91 minutes fall on 18 September.

**The checking boundary.** Checking is the CHECK code alone: 21 rows and 91 minutes in 19 case-days on all five days, 37 minutes in Ordering (checks before ordering) and 54 in Monitoring (checks of confirmations after ordering). The notes name a price or a delivery date for 7 rows and 33 minutes and an unspecified mismatch for 1 row and 20 minutes; for the other 13 rows (38 minutes) the compared field is blank or generic. The observer's clarification that those checks concerned price and delivery date supports relevance but is observer knowledge, not a recorded field. The proposed support is the comparison of supplier quotations before ordering and confirmations after ordering against the applicable PO version; quantity and missing-line comparison are additional proposed fields without measured durations.

**Interpretation.** The figures locate recorded work; they are not validated eligible-task totals or savings. Candidates overlap (every timed maximalisatie row is a PO row; part of the article-data rows are Specification rows) and must not be summed. The process-level candidate "standard / review / manual" uses the PO code as its closest measured area and does not measure an entire routing intervention. The gated request-intake scenario is narrowed to finding old POs and drawings; the raw codes do not isolate that search, so the table gives the whole Specification stage and, below it, the six rows (36 minutes, five case-days on three days) whose note names such a search. Dennis's separate evidence informs business relevance, not Arno's workload denominator.

<a id="feasibility"></a>
## 2. Feasibility before scoring

**Purpose and approach.** A weighted preference cannot compensate for a failed requirement, so six gates are applied to each candidate before any scoring, using the evidence of the [Stage A worksheet](https://github.com/Luuuuuig/Bachelor-End-Project/blob/main/docs/analyze/Stage_A_Test_Account_Check_2026-10-07.xlsx). Verdicts are **Pass** (supported), **Provisional pass** (expected, with an unresolved check), **Redesign** (only a defined narrower form may continue) and **Reject** (removed from scoring). The two cells hold the gate evidence per candidate; the detailed reasons and check IDs are displayed in [Appendix A](#gate-evidence).

| Gate | Requirement |
|---|---|
| G1 Permission | Data and tools are permitted for this research use |
| G2 Suitable accessible data | Relevant linked inputs are available for a credible prototype and test |
| G3 Reference answers and manual comparator | Reproducible correct answers and comparable manual checks can be established for SQ5 |
| G4 Controlled risks and acceptable human review | Errors can be detected or reversed, with buyer acceptance, review, traceability and fallback |
| G5 Feasible development and evaluation | Access, build effort and buyer testing fit the BEP |
| G6 Meaningful AI contribution | AI has a useful role beyond simpler rules, search or extraction |

In [6]:
# Stage A for Order and confirmation checking: gate, verdict, reason, check that settles it
gate_rows = []

candidate = 'Order and confirmation checking'
gate_rows += [
    (candidate, 1, 'Pass',
     "Johan's permission covers using the test-account data and processing its documents "
     'with an external AI service (checks G1–G2, confirmed by Yijie on 7 October). What '
     'must be anonymised in the thesis (G3) sets how results are reported, not whether the'
     ' data may be used.',
     ''),
    (candidate, 2, 'Provisional pass',
     'PO lines are standard Exact records and the observations provide examples. Whether '
     'matching confirmations/quotations exist and represent the target work remains '
     'unverified. Inspect their period and current-practice differences. If unsuitable, '
     'demonstrate a permitted representative alternative before final selection; the '
     'provisional scores do not close this gate.',
     'PC1–PC3, G4, G5'),
    (candidate, 3, 'Provisional pass',
     'The proposed reference is the approved price per line. The buyer must verify its '
     'correctness, version and applicability to the item, quantity and order date. '
     'Availability and traceability are not yet established. Demonstrate a trustworthy '
     'source before final selection. Constructing test workloads does not supply their '
     'correct answers. For delivery date, quantity and line presence, a versioned PO '
     'is a comparison input, not automatic proof of the accepted business answer. '
     'Validate document fields, discrepancy labels and accepted actions separately.',
     'PC4, PC6'),
]

# Stage A for Order and confirmation checking, continued: gates 4 to 6
gate_rows += [
    (candidate, 4, 'Provisional pass',
     'The function only flags differences; the buyer judges each one and nothing changes '
     "in Exact without the buyer. On 8 October Yijie reported objection to automatic "
     "PO creation and email sending. This does not establish acceptance or rejection "
     "of flagging. Arno's acceptance of the specific support remains open.",
     'Arno follow-up guide, part D (rows 5–6)'),
    (candidate, 5, 'Provisional pass',
     'A prototype on exported PO lines and supplier documents could fit the BEP without '
     'Exact integration. This depends on successful exports, matched documents and buyer '
     'time for manual comparison; these checks remain open.',
     'G6, PC1, PC7'),
    (candidate, 6, 'Provisional pass',
     'Notes describe paper, PDF, website and email prices with differing item '
     'descriptions. This may justify AI-assisted reading/matching. Inspect actual samples '
     'and compare the proposed role with structured data, OCR, search and deterministic '
     'rules. The Tater et al. precedent still requires full-text review and does not '
     'establish added value here.',
     'PC2'),
]

In [7]:
# Stage A for Article / supplier data: gate, verdict, reason, check that settles it
candidate = 'Article / supplier data'
gate_rows += [
    (candidate, 1, 'Pass',
     "Johan's permission covers using the test-account data and processing its documents "
     'with an external AI service (checks G1–G2, confirmed by Yijie on 7 October).',
     ''),
    (candidate, 2, 'Provisional pass',
     'Article records exist in Exact; whether the supplier documents used to create them '
     'are linked is not yet known.',
     'AR2'),
    (candidate, 3, 'Provisional pass',
     'Existing article records are candidate references for extracted fields, subject to '
     'checking against source documents and buyer-approved values. Do not assume '
     'historical master data are correct.',
     'AR1'),
    (candidate, 4, 'Pass',
     'The buyer checks the fields before creating the article.',
     ''),
    (candidate, 5, 'Provisional pass',
     'A bounded prototype could use exported article records and supplier documents if '
     'exports and matching inputs are available; that availability remains to be checked.',
     'G6, AR2'),
    (candidate, 6, 'Provisional pass',
     'Different supplier-document layouts may justify AI-assisted field extraction. '
     'Inspect actual samples and compare with simpler extraction/rules; the cited '
     'benchmark is a research lead, not demonstrated performance in this setting.',
     ''),
]

# Stage A for Maximalisatie: gate, verdict, reason, check that settles it
candidate = 'Maximalisatie'
gate_rows += [
    (candidate, 1, 'Provisional pass',
     "Johan's permission covers using the test-account data and processing its documents "
     'with an external AI service (checks G1–G2, confirmed by Yijie on 7 October). Whether'
     ' demand data in Orbis are also needed and permitted is not yet known.',
     'MX4'),
    (candidate, 2, 'Redesign',
     'Full decision support needs past stock, advice and lead-time states, which a test '
     'copy is unlikely to hold. Narrowed output: when a PO is created, a list of the open '
     'POs and pending demand for the same supplier; the buyer decides whether to combine '
     'or hold.',
     'MX1, MX2'),
    (candidate, 3, 'Redesign',
     'No validated reference for a good hold-or-order decision has been demonstrated, so '
     'the proposed redesign excludes that decision. Lines later added to a PO are '
     'candidate examples for related-demand retrieval; the buyer must check relevance and '
     'omissions before they serve as reference answers.',
     'MX3'),
    (candidate, 4, 'Pass',
     'A suggestion only; the buyer decides.',
     ''),
    (candidate, 5, 'Provisional pass',
     'The narrowed version fits the BEP if open POs and demand can be exported; the full '
     'version depends on Exact and Orbis data.',
     'G6, MX4'),
    (candidate, 6, 'Provisional pass',
     'Listing same-supplier open POs and demand is close to a rule. Matching free-text '
     'demand may justify AI only if actual examples show useful support beyond simple '
     'search/rules.',
     ''),
]

# Stage A for Request intake & validation: gate, verdict, reason, check that settles it
candidate = 'Request intake & validation'
gate_rows += [
    (candidate, 1, 'Provisional pass',
     "Johan's permission covers using the test-account data and processing its documents "
     'with an external AI service (checks G1–G2, confirmed by Yijie on 7 October). Access '
     'to the drawing archive is not yet known.',
     'RI2'),
    (candidate, 2, 'Redesign',
     'Requests arrive by email, at the desk or by phone and are probably not stored in '
     'Exact. Narrowed output: for the item being ordered, the old POs and the drawings '
     'found in Exact or the drawing archive, for the buyer to use.',
     'RI1, RI2'),
    (candidate, 3, 'Redesign',
     'No validated list of required drawings or complete-request definition is available '
     'for completeness assessment, so the proposed redesign excludes it. Previously used '
     'POs/drawings are candidate retrieval references; the buyer must verify their '
     'relevance, revision and completeness before evaluation.',
     'RI3'),
    (candidate, 4, 'Pass',
     'Retrieval and flags only; the buyer decides.',
     ''),
    (candidate, 5, 'Provisional pass',
     'Depends on access to where drawings are kept.',
     'RI2'),
    (candidate, 6, 'Provisional pass',
     'Free-text item descriptions may justify AI-assisted retrieval of POs/drawings. '
     'Inspect actual examples and establish whether it adds useful support beyond ordinary'
     ' search and rules.',
     ''),
]

# Stage A for Standard / review / manual: gate, verdict, reason, check that settles it
candidate = 'Standard / review / manual'
gate_rows += [
    (candidate, 1, 'Pass',
     "Johan's permission covers using the test-account data and processing its documents "
     'with an external AI service (checks G1–G2, confirmed by Yijie on 7 October).',
     ''),
    (candidate, 2, 'Provisional pass',
     'PO records exist, but nothing marks a case as standard, review or manual.',
     'SR1'),
    (candidate, 3, 'Redesign',
     'No reference standard exists without the buyer labelling past cases.',
     'SR1'),
    (candidate, 4, 'Redesign',
     "The automatic creation-and-sending arrangement conflicts with the buyers' objection "
     'reported on 8 October. A bounded review aid requires its own acceptance check.',
     'Arno follow-up guide, part D'),
    (candidate, 5, 'Reject',
     'As a stand-alone candidate, an end-to-end routing redesign with Exact integration '
     'does not fit the BEP. It stays the design frame for the chosen task: cases without a'
     ' flag are standard, flagged cases go to review.',
     ''),
    (candidate, 6, 'Provisional pass',
     'Routing can largely be rules; AI fits only specific steps.',
     ''),
]

# Stage A for PO supplier communication: gate, verdict, reason, check that settles it
candidate = 'PO supplier communication'
gate_rows += [
    (candidate, 1, 'Pass',
     "Johan's permission covers using the test-account data and processing its documents "
     'with an external AI service (checks G1–G2, confirmed by Yijie on 7 October).',
     ''),
    (candidate, 2, 'Provisional pass',
     'Sent PO emails are probably retrievable.',
     'CM1'),
    (candidate, 3, 'Provisional pass',
     'A buyer-verified recipient, attachment and message could provide a reference. A '
     'historically sent email alone does not establish correctness; validate reference '
     'examples if this supporting opportunity is pursued.',
     ''),
    (candidate, 4, 'Provisional pass',
     'On 8 October Yijie reported opposition to automatic PO creation and email sending '
     'as one arrangement. Acceptance of a separate draft or send function still needs '
     'its own check; do not assume consent to automatic sending.',
     'Arno follow-up guide, part D (row 11)'),
    (candidate, 5, 'Pass',
     'Small and buildable within the BEP.',
     ''),
    (candidate, 6, 'Reject',
     'Forwarding a generated PO with a standard message is a deterministic step; ordinary '
     'automation is enough. Kept as a quick-win recommendation.',
     ''),
]

In [8]:
gates = pd.DataFrame(gate_rows, columns=['candidate', 'gate', 'verdict', 'reason', 'check'])
gate_table = gates.pivot(index='candidate', columns='gate', values='verdict')
gate_table.columns = [f'Gate {g}' for g in gate_table.columns]
severity = {'Pass': 0, 'Provisional pass': 1, 'Redesign': 2, 'Reject': 3}
labels = {0: 'Pass', 1: 'Provisional pass', 2: 'Redesign', 3: 'Reject'}
gates['severity'] = gates['verdict'].map(severity)
stage_a = gates.groupby('candidate')['severity'].max().map(labels).to_frame('stage_a_result')
stage_a['scored_in_stage_b'] = stage_a['stage_a_result'] != 'Reject'
# One verdict table; the final column shows whether the defined scenario is shortlisted.
gate_view = gate_table.join(stage_a["stage_a_result"].rename("Overall verdict"))
gate_view["Scoring scenario"] = stage_a["scored_in_stage_b"].map({True:"Retained provisionally", False:"Excluded"})
display(gate_view)

,Gate 1,Gate 2,Gate 3,Gate 4,Gate 5,Gate 6,Overall verdict,Scoring scenario
candidate,,,,,,,,
Article / supplier data,Pass,Provisional pass,Provisional pass,Pass,Provisional pass,Provisional pass,Provisional pass,Retained provisionally
Maximalisatie,Provisional pass,Redesign,Redesign,Pass,Provisional pass,Provisional pass,Redesign,Retained provisionally
Order and confirmation checking,Pass,Provisional pass,Provisional pass,Provisional pass,Provisional pass,Provisional pass,Provisional pass,Retained provisionally
PO supplier communication,Pass,Provisional pass,Provisional pass,Provisional pass,Pass,Reject,Reject,Excluded
Request intake & validation,Provisional pass,Redesign,Redesign,Pass,Provisional pass,Provisional pass,Redesign,Retained provisionally
Standard / review / manual,Pass,Provisional pass,Redesign,Redesign,Reject,Provisional pass,Reject,Excluded


**Result.** Two candidates are rejected: PO supplier communication at G6, because forwarding a generated PO with a standard message can use ordinary automation, and standard / review / manual at G5, because an end-to-end redesign does not fit the BEP (it remains a possible presentation frame for the focal task). Four candidates go to the scoring as provisional scenarios: checking and article/supplier data with open data and reference gates (G2, G3), maximalisatie narrowed to listing same-supplier open POs and pending demand, and request intake narrowed to finding earlier POs and drawings.

**Interpretation.** The four are scenarios, not closed gate decisions. For checking, G2 and G3 require paired documents, applicable versions and buyer-validated references (PC1 to PC4), G4 requires Arno's acceptance of this specific aid, G5 workable exports and buyer testing, and G6 actual documents and a comparison with simpler methods. Permission for test-account data and external AI processing (7 October) establishes none of these, and a "Pass" on G4 for the low-autonomy alternatives describes proposed controls, not demonstrated buyer endorsement. When a required minimum fails, the gate is reopened instead of lowering a score.

<a id="criteria"></a>
## 3. Criteria, anchors and weights

**Purpose and approach.** Seven criteria connect the workload objective to a buildable, evaluable intervention. Scores run from 1 to 5; 2 and 4 represent evidence between the stated anchors. The criteria and cut-offs are project judgements informed by multi-criteria decision analysis and task-technology fit, not numbers prescribed by those sources. The weights are swing weights: the most important swing from worst to best gets 100 and the others are set relative to it.

| Criterion | Definition and rationale | Safeguard against double counting |
|---|---|---|
| Observed workload contribution | Arno's current effort and difficulty within the bounded activity; the primary objective | Observed records, 560-minute denominator; no assumed savings or sum of overlapping areas |
| Business value beyond counted workload | Purchasing or company consequences beyond buyer time | Exclude time and its wage equivalent; a discrepancy amount is not an achieved benefit |
| AI–task fit | Suitability of a useful AI function beyond simpler methods | Distinguish task suitability from build effort and acceptance |
| Data readiness | Usability of permitted inputs above the gate minimum | Input preparation is distinct from engineering and test arrangements |
| Repetition / process standardisation | Recurrence and stable inputs, outputs and exception boundaries | Do not count recurrence as a second time-saving benefit |
| Prototype feasibility | Engineering effort and dependencies of the bounded build | Separate integration and build effort from data cleaning and testing |
| Evaluation feasibility | A defensible manual-versus-assisted comparison for SQ5 | Constructed cases still need correct answers, comparable procedures and buyer time |

**Anchors.** The score-5 anchor for business value requires the company supervisor's confirmation; it is not relaxed.

| Criterion | 1 | 3 | 5 |
|---|---|---|---|
| Observed workload contribution | Under 5% of in-scope minutes, little recorded burden | 5–15%, or less time with clear recorded burden | Over 15%, with recorded burden on several days |
| Business value beyond counted workload | No effect named beyond buyer time | One effect named with an example | Effect confirmed by the company supervisor with examples or figures |
| AI–task fit | Just passes gate 6: AI adds little beyond rules | AI needed for one part, the rest is rules | The core of the task needs reading unstructured documents or judging patterns, with a precedent in the literature |
| Data readiness above the gate minimum | Few hand-collected examples, no recorded correct outcome | Enough past cases with moderate cleaning or labelling | Representative past cases can be exported, with a traceable correct outcome |
| Repetition / process standardisation | Seen on fewer than 3 of 5 days, or inputs differ every time | Every day, but inputs vary or exceptions are frequent | Many cases every day, stable inputs and outputs, clear exception boundary |
| Prototype / technical feasibility | Needs Exact integration or access not yet granted | Can be built on exports or copies, with manual steps | Can be built on available data and tools, with no integration |
| Evaluation feasibility | No reference answer, or no comparable cases | Some cases and a rubric; limited buyer time | Enough repeated cases with a reference answer, and buyer time, or test workloads built from past orders |

A failed gate cannot be turned into a score of 1. The score-5 anchor for business value requires the company supervisor's confirmation ([Appendix B](#score-evidence), boundary notes). The additive model still assumes that trade-offs are useful despite possible residual criterion overlap.

**Swing weighting.** Compare the value of moving a candidate from 1 to 5 on each criterion. Assign 100 to the most valuable swing and express the others relative to it; divide each swing by the sum. The existing swings below are **Yijie's provisional weights**, recorded on 7 October before scoring, though workload figures were already known. Zhongxin agreed to review weights on 8 October; she has not supplied or endorsed a final set. Independent elicitation before showing the scores, then discussing/combining sets, is an existing proposal, not a confirmed requirement.

In [9]:
# Yijie's provisional swings of 7 October (docs/analyze/Swing_Weights_Yijie_2026-10-07.md); not Zhongxin's
weights = pd.DataFrame([('workload', 'Observed workload contribution', 100), ('business_value', 'Business value beyond counted workload', 40), ('ai_fit', 'AI–task fit', 50), ('data_readiness', 'Data readiness above the gate minimum', 60), ('repetition', 'Repetition / process standardisation', 30), ('prototype', 'Prototype / technical feasibility', 40), ('evaluation', 'Evaluation feasibility', 50)], columns=['criterion', 'label', 'swing']).set_index('criterion')
weights['weight'] = weights['swing'] / weights['swing'].sum()
weight_reasons = {
    "workload": "Primary objective: reduce Arno's workload",
    "business_value": "Company effects, secondary to the workload objective",
    "ai_fit": "A useful AI contribution beyond the gate minimum",
    "data_readiness": "Input quality supports both build and test",
    "repetition": "Recurrence already partly contributes to workload",
    "prototype": "Build feasibility above the minimum gate",
    "evaluation": "A credible comparison is needed for SQ5",
}
weight_view = weights.set_index("label").rename(columns={"swing":"Swing", "weight":"Weight (%)"})
weight_view["Weight (%)"] *= 100
weight_view["Rationale"] = [weight_reasons[c] for c in weights.index]
display(weight_view.style.format({"Swing":"{:.0f}", "Weight (%)":"{:.1f}"}))

,Swing,Weight (%),Rationale
label,,,
Observed workload contribution,100,27.0,Primary objective: reduce Arno's workload
Business value beyond counted workload,40,10.8,"Company effects, secondary to the workload objective"
AI–task fit,50,13.5,A useful AI contribution beyond the gate minimum
Data readiness above the gate minimum,60,16.2,Input quality supports both build and test
Repetition / process standardisation,30,8.1,Recurrence already partly contributes to workload
Prototype / technical feasibility,40,10.8,Build feasibility above the minimum gate
Evaluation feasibility,50,13.5,A credible comparison is needed for SQ5


**Result.** Workload has the largest swing (100) and a weight of 27.0%; data readiness follows (60, 16.2%), then AI fit and evaluation feasibility (50 each, 13.5%). The full-precision weights are used throughout; the displayed percentages may not sum to exactly 100 after rounding.

**Interpretation.** These are Yijie's provisional weights, set on 7 October after he had seen the workload figures; they are not Zhongxin's judgement. She can set her own swings for the seven criteria (option A, an informed review since this notebook shows the ranking) or review Yijie's and say what to change (option B). The agreed set is then recorded with the date and the totals rerun.

<a id="ranking"></a>
## 4. Scores, ranking and interpretation

**Purpose and approach.** Score the four shortlisted scenarios on the seven criteria, with a low and a high value for the uncertainty range and the evidence behind each score. The scores are the 7 October inputs re-examined on 12 October against the raw-code figures of the Arno analysis; the evidence strings cite them. All scores are Yijie's provisional judgement until Zhongxin's review. WL = workload, BV = business value, AI = AI–task fit, Data = data readiness, Rep = repetition, Build = prototype feasibility, Eval = evaluation feasibility.

In [10]:
# Scores for Order and confirmation checking: criterion, score, low, high, evidence (raw-code figures, Arno analysis part 5 and 8)
score_rows = []

candidate = 'Order and confirmation checking'
score_rows += [
    (candidate, 'workload', 5, 4, 5,
     'All CHECK rows: 91 of 560 minutes (16.2%) in 19 case-days on all five days; 57 minutes '
     '(62.6%) on 18 September and 8.3% of the improvement minutes without that day; recorded '
     'difficulties on 31 Aug, 1 and 18 Sep (missing delivery date, blocked check, a mismatch '
     'difficult to explain, a large price difference) [Arno analysis 5.1, 5.2]. The notes name '
     'a price or delivery date for 7 rows / 33 minutes and an unspecified mismatch for 1 row / '
     '20 minutes; the 7 October observer clarification covers the other 38 minutes but is not '
     'a recorded field. Low end: the dependence on 18 September.'),
    (candidate, 'business_value', 3, 3, 4,
     'Dennis reported Sales losses from outdated purchase prices (15 September); buyers '
     'lack time for updates (2 October), repeated at the 8 October meeting. Qualitative '
     'business relevance is supported. Financial impact and achievable benefit remain '
     'unquantified; Johan\'s confirmation with examples or figures is still to come.'),
    (candidate, 'ai_fit', 4, 3, 4,
     'Reading prices and lines from paper, PDF, website and e-mail documents and matching '
     'them to PO lines is the core of the task (8 Sep row 21 compares a price against the '
     'quotation and the website); precedent Tater et al. (2022) [AI table rows 5–7]. Low end: '
     'if documents are structured, only matching needs AI.'),
    (candidate, 'data_readiness', 3, 2, 4,
     'PO lines are standard Exact data; confirmations and approved prices are expected but'
     ' not yet seen in the test account [checks PC1–PC4]. If they are missing, gates 2–3 '
     'reopen; this score does not absorb that.'),
    (candidate, 'repetition', 3, 3, 4,
     'Every day, 19 case-days; the task is stable (compare price and date per line) but '
     'inputs vary in format and size, 1 to 29 lines in the usable checks [Arno analysis 5.3].'),
    (candidate, 'prototype', 3, 3, 4,
     'Proposed route: exports and documents with manual steps, without integration; '
     'not yet demonstrated [check G6].'),
    (candidate, 'evaluation', 4, 3, 5,
     "Proposed reference per line (a buyer-validated approved price) and Zhongxin's fallback of test "
     'workloads from past orders; the observed time per line is imprecise (0.87 minutes per '
     'line pooled, 0.90 on 18 September against 0.83 on the other days; mean rate per check '
     '1.24, 95% CI 0.70 to 1.78), so the time baseline is weak [Arno analysis 5.3].'),
]

# Scores for Article / supplier data: criterion, score, low, high, evidence
candidate = 'Article / supplier data'
score_rows += [
    (candidate, 'workload', 3, 3, 3,
     '43 of 560 minutes (7.7%) in 13 timed rows (17 ledger rows) on four days, 6 case-days; '
     'the 31 August article-master work could not be finished because required data were '
     'missing and continued on 1 September (row 33) [Arno analysis part 8 ledger].'),
    (candidate, 'business_value', 1, 1, 2,
     'No effect named beyond buyer time.'),
    (candidate, 'ai_fit', 4, 3, 4,
     'Extracting item fields from supplier documents in different layouts is the core of '
     'the task; precedent Šimsa et al. (2023), a benchmark [AI table row 4].'),
    (candidate, 'data_readiness', 3, 2, 4,
     'Article records exist; whether their source documents are linked is not yet known '
     '[checks AR1, AR2].'),
    (candidate, 'repetition', 2, 2, 3,
     'Four of five days, 6 case-days; item data differ per article.'),
    (candidate, 'prototype', 3, 2, 4,
     'Can run on exported article records and supplier documents [checks G6, AR2].'),
    (candidate, 'evaluation', 3, 2, 4,
     'Existing article records can be the reference; few cases per week [check AR3].'),
]

In [11]:
# Scores for Maximalisatie: criterion, score, low, high, evidence
candidate = 'Maximalisatie'
score_rows += [
    (candidate, 'workload', 3, 3, 3,
     '73 of 560 minutes (13.0%) in 29 timed rows on all five days, 26 case-days [Arno analysis '
     'part 8 ledger]; 28 of those minutes are rows where MAX is named within a PO interval and '
     '45 are rows where lines were added without a recorded search (5.0% and 8.0% of 560). The '
     'list supports finding demand to combine; the decision time itself cannot be separated.'),
    (candidate, 'business_value', 3, 2, 3,
     'One effect named: each PO costs time and money (Dennis via Yijie, 2 October); '
     'example of 4 lines added to a 12-line PO (31 Aug row 52, OBS-15).'),
    (candidate, 'ai_fit', 2, 1, 3,
     'The narrowed version is close to a rule (same supplier, open PO); AI only for '
     'matching demand descriptions [AI table row 8].'),
    (candidate, 'data_readiness', 2, 1, 3,
     'Past advice and stock states are unlikely to be stored; open POs probably are '
     '[checks MX1–MX3].'),
    (candidate, 'repetition', 3, 3, 3,
     'Every day, 26 case-days in the ledger; inputs vary with stock, urgency and order value '
     '[Arno analysis part 8].'),
    (candidate, 'prototype', 2, 1, 3,
     'Needs open-PO and demand data, possibly Orbis [check MX4].'),
    (candidate, 'evaluation', 3, 2, 3,
     'The narrowed list can be checked against the lines later added to the same open PO '
     '[check MX3]; how many such cases the account holds is not yet known.'),
]

# Scores for Request intake & validation: criterion, score, low, high, evidence
candidate = 'Request intake & validation'
score_rows += [
    (candidate, 'workload', 3, 3, 5,
     'Raw definition: the Specification stage, 121 of 560 minutes (21.6%) in 33 records, 23 '
     'case-days, all five days; 108 minutes are CLAR rows with a note [Arno analysis part 8]. '
     'The gated scenario is narrowed to finding old POs and drawings, and the raw codes do not '
     'isolate that search: the six Specification rows whose note names a drawing, an old-PO '
     'lookup or an item history hold 36 minutes (6.4%, the score-3 band) on three days. Scored '
     'on the narrowed scenario; the high of 5 is the whole stage (over 15%), which would only '
     'count if the retrieval output supported most of it.'),
    (candidate, 'business_value', 2, 1, 3,
     'Incomplete drawing sets from Engineering delay orders (18 Sep OBS-10: drawings looked '
     'for, asked for and checked, rows 27, 28 and 41); no business effect beyond buyer time '
     'is named.'),
    (candidate, 'ai_fit', 3, 2, 4,
     'Finding the right old PO or drawing from free-text item descriptions needs AI for '
     'the search; weak precedent (Lombardi et al., 2025, other drawing domain) [AI table '
     'row 1].'),
    (candidate, 'data_readiness', 2, 1, 3,
     'Old POs are in Exact; where drawings are kept and how they link to items is unknown '
     '[checks RI2, RI3].'),
    (candidate, 'repetition', 2, 1, 3,
     'The retrieval-like notes fall on three of five days (5 case-days) with varying inputs: '
     'between the 1 and 3 anchors. The whole Specification stage occurs every day (23 '
     'case-days) with varying inputs, which sets the high end.'),
    (candidate, 'prototype', 2, 1, 3,
     "Depends on access to the drawing archive and on searching Exact's item history "
     '[check RI2].'),
    (candidate, 'evaluation', 3, 2, 3,
     'The old PO or drawing that was actually used is the reference; few cases were '
     'observed (six rows with a retrieval-like note in 5 case-days: one old-PO lookup, four '
     'rows about drawings and one item history).'),
]

In [12]:
scores = pd.DataFrame(score_rows, columns=['candidate', 'criterion', 'score', 'low', 'high', 'evidence'])
scored = scores.loc[scores['candidate'].isin(stage_a.index[stage_a['scored_in_stage_b']])]
score_table = scored.pivot(index='candidate', columns='criterion', values='score')[weights.index]
criterion_short = {"workload":"WL", "business_value":"BV", "ai_fit":"AI", "data_readiness":"Data",
                   "repetition":"Rep", "prototype":"Build", "evaluation":"Eval"}
display(score_table.rename(columns=criterion_short))

criterion,WL,BV,AI,Data,Rep,Build,Eval
candidate,,,,,,,
Article / supplier data,3,1,4,3,2,3,3
Maximalisatie,3,3,2,2,3,2,3
Order and confirmation checking,5,3,4,3,3,3,4
Request intake & validation,3,2,3,2,2,2,3


For candidate $i$, the weighted preference total is

$$T_i=100\sum_{j=1}^{7} w_j\frac{s_{ij}}{5},\qquad \sum_j w_j=1.$$

Each score is multiplied by its full-precision weight. All scores of 5 give 100; all scores of 1 give 20. Scores are **preferences, not expected savings or probabilities of success**.

In [13]:
totals = (score_table @ weights['weight'] / 5 * 100).sort_values(ascending=False).to_frame('total')
totals['lead_over_next'] = totals['total'] - totals['total'].shift(-1)
ranking = totals[["total"]].rename(columns={"total": "Weighted preference score"})
ranking.insert(0, "Rank", range(1, len(ranking) + 1))
display(ranking.style.format({"Weighted preference score": "{:.1f}"}))

,Rank,Weighted preference score
candidate,,
Order and confirmation checking,1,76.2
Article / supplier data,2,56.8
Maximalisatie,3,51.9
Request intake & validation,4,50.8


**Result.** Checking 76.2, article/supplier data 56.8, maximalisatie 51.9, request intake 50.8. Checking scores highest or joint highest on every criterion, so its lead of 19.5 points is not driven by one weight. No score changed in the 12 October rebuild: every raw figure sits in the same anchor band as the figure it replaced. Two ranges widened: the request-intake highs went from 3 to 5 (workload) and from 2 to 3 (repetition), because its raw definition (the whole Specification stage, 121 minutes) is broader than the narrowed retrieval scenario that is scored (36 note-named minutes on three days); [Appendix B](#score-evidence) lists every score and range before and after with the reason. If the whole stage counted, request intake would score 4 and 3 on those criteria and reach 57.8, still 18.4 points behind checking (the broad-proxy scenario of Section 5).

**Interpretation.** Checking's strongest advantage is observed workload; its evaluation advantage assumes that suitable references and testing cases can be established. Article/supplier data has a strong AI-fit score but no named business effect beyond buyer time; the narrowed maximalisatie list is close to rules; request intake depends on access to drawings and old POs. Retain checking as the provisional leader and test the uncertainty next; its score does not establish feasible data, acceptance, eligible workload or benefit.

<a id="robustness"></a>
## 5. Robustness and uncertainty

**Purpose and approach.** Which uncertain judgements could change the selection? Seven checks on the same preference-point scale: equal weights; each weight multiplied by 0.8 or 1.2 and renormalised; each score changed by one point within 1 to 5, one at a time; checking's workload score alone set to 3; request intake scored on the whole Specification stage (workload 4, repetition 3); combined one-point changes inside the recorded low/high ranges, from the largest weighted change down, to find the fewest that overturn the gap; and the joint stress case (checking at its lows, every challenger at its highs). The detailed outputs and a random-draw illustration are in [Appendix C](#sensitivity-detail).

In [14]:
low = scored.pivot(index='candidate', columns='criterion', values='low')[weights.index]
high = scored.pivot(index='candidate', columns='criterion', values='high')[weights.index]
leader = totals.index[0]

equal = pd.Series(1 / len(weights), index=weights.index)
equal_totals = (score_table @ equal / 5 * 100).sort_values(ascending=False)

rows = []
for criterion in weights.index:
    for change in [0.8, 1.2]:
        w = weights['weight'].copy()
        w[criterion] *= change
        w = w / w.sum()
        t = (score_table @ w / 5 * 100).sort_values(ascending=False)
        rows.append([criterion, f'{change - 1:+.0%}', t.index[0], t.iloc[0] - t.iloc[1]])
weight_checks = pd.DataFrame(rows, columns=['criterion', 'weight_change', 'leader', 'lead_over_next'])

rows = []
for candidate in score_table.index:
    for criterion in score_table.columns:
        for change in [-1, 1]:
            s = score_table.copy()
            s.loc[candidate, criterion] = min(5, max(1, s.loc[candidate, criterion] + change))
            t = (s @ weights['weight'] / 5 * 100).sort_values(ascending=False)
            rows.append([candidate, criterion, change, t.index[0], t.iloc[0] - t.iloc[1]])
score_checks = pd.DataFrame(rows, columns=['candidate', 'criterion', 'change', 'leader', 'lead_over_next'])

checking_workload_3 = score_table.copy()
checking_workload_3.loc['Order and confirmation checking', 'workload'] = 3
checking_workload_3_totals = (checking_workload_3 @ weights['weight'] / 5 * 100).sort_values(ascending=False)


# Broad-proxy scenario: request intake scored on the whole Specification stage (workload 4, repetition 3) instead of the narrowed retrieval scenario
broad_proxy = score_table.copy()
broad_proxy.loc['Request intake & validation', ['workload', 'repetition']] = [4, 3]
broad_proxy_totals = (broad_proxy @ weights['weight'] / 5 * 100).sort_values(ascending=False)

In [15]:
# Combined changes within the recorded ranges, and the joint low/high stress case
rows = []
for challenger in score_table.index.drop(leader):
    steps = []
    for c in score_table.columns:
        points = weights.loc[c, 'weight'] * 20
        steps += [[f'{leader}: {c} -1', points]] * int(score_table.loc[leader, c] - low.loc[leader, c])
        steps += [[f'{challenger}: {c} +1', points]] * int(high.loc[challenger, c] - score_table.loc[challenger, c])
    steps = pd.DataFrame(steps, columns=['change', 'points']).sort_values('points', ascending=False, kind='stable')
    steps['gap_closed'] = steps['points'].cumsum()
    gap = totals.loc[leader, 'total'] - totals.loc[challenger, 'total']
    if steps['gap_closed'].max() > gap:
        n = int((steps['gap_closed'] <= gap).sum()) + 1
        rows.append([challenger, gap, len(steps), n, '; '.join(steps['change'].head(n))])
    else:
        rows.append([challenger, gap, len(steps), np.nan, 'not possible within the ranges'])
combined = pd.DataFrame(rows, columns=['challenger', 'gap_points', 'changes_available', 'fewest_changes_needed', 'those_changes'])

stress = high.copy()
stress.loc[leader] = low.loc[leader]
stress_totals = (stress @ weights['weight'] / 5 * 100).sort_values(ascending=False)

In [16]:
def top_two_text(series):
    return f"{series.iloc[0]:.1f} vs {series.iloc[1]:.1f}; lead {series.iloc[0]-series.iloc[1]:.1f}"
combined_possible = combined.loc[combined["fewest_changes_needed"].notna()]
robustness_summary = pd.DataFrame([
    ["Alternative weights: equal", equal_totals.index[0], top_two_text(equal_totals)],
    ["Alternative weights: each ±20%", f"Checking in {(weight_checks['leader']==leader).sum()}/{len(weight_checks)} cases",
     f"Smallest lead {weight_checks['lead_over_next'].min():.1f}"],
    ["Individual scores: each ±1", f"Checking in {(score_checks['leader']==leader).sum()}/{len(score_checks)} cases",
     f"Smallest lead {score_checks['lead_over_next'].min():.1f}"],
    ["Checking workload = 3 only", checking_workload_3_totals.index[0], top_two_text(checking_workload_3_totals)],
    ["Request intake on the whole Specification stage (4, 3)", broad_proxy_totals.index[0], top_two_text(broad_proxy_totals)],
    ["Combined changes within ranges", combined_possible.loc[combined_possible["fewest_changes_needed"].idxmin(), "challenger"],
     f"Can overtake after {int(combined_possible['fewest_changes_needed'].min())} one-point changes"],
    ["Joint low/high stress case", stress_totals.index[0], top_two_text(stress_totals)],
], columns=["Check", "Leader / result", "Preference-point comparison"])
display(robustness_summary.set_index("Check"))

,Leader / result,Preference-point comparison
Check,,
Alternative weights: equal,Order and confirmation checking,71.4 vs 54.3; lead 17.1
Alternative weights: each ±20%,Checking in 14/14 cases,Smallest lead 18.3
Individual scores: each ±1,Checking in 56/56 cases,Smallest lead 14.1
Checking workload = 3 only,Order and confirmation checking,65.4 vs 56.8; lead 8.6
"Request intake on the whole Specification stage (4, 3)",Order and confirmation checking,76.2 vs 57.8; lead 18.4
Combined changes within ranges,Article / supplier data,Can overtake after 6 one-point changes
Joint low/high stress case,Request intake & validation,73.5 vs 68.6; lead 4.9


**Result.** Checking leads every single-change check: all 14 weight changes (smallest lead 18.3 points) and all 56 attempted one-point score changes (smallest lead 14.1; two of the 56 are clipped at the ends of the scale and change nothing). With its workload score reduced to 3 it scores 65.4 against 56.8, a lead of 8.6; with request intake on the whole Specification stage the lead is 18.4. Combined changes can reverse the ranking: article/supplier data overtakes after six one-point changes (checking down one on workload, data readiness, AI fit and evaluation feasibility; article data up one on data readiness and evaluation feasibility), request intake after seven, maximalisatie not within its ranges. In the joint stress case request intake leads at 73.5, ahead of article/supplier data (68.6) and checking (62.2). In 10,000 random draws within the ranges checking wins 9,933.

**Interpretation.** The ranking is robust to any single judgement but not to a cluster of unfavourable ones, and the stress case shows that the recorded ranges are wide enough to reverse it. The judgements that matter most are checking's data readiness, AI fit and evaluation feasibility, which is why the next evidence to collect is paired documents and reference answers, then AI versus simpler methods and the evaluation arrangements. The random-draw frequencies illustrate the assumed ranges; they are not probabilities of success. If data or references fail the minimum, the gate is reopened rather than a score adjusted.

<a id="recommendation"></a>
## 6. Recommendation and remaining decisions

**Provisional recommendation.** Develop the selection case for bounded **reading, matching and flagging assistance**: read agreed fields with source references, suggest line matches, display differences and unreadable or unmatched material, and let Arno decide whether to accept, correct or clarify. Numeric comparisons can use explicit rules; the AI role must be justified by actual document layouts or ambiguous descriptions. Arno keeps the purchasing decisions, the system changes and the sending; an unflagged result is not an authorisation. The buyers' reported opposition to automatic PO creation and sending does not settle their acceptance of this review aid.

| # | Remaining decision or input | Evidence required and consequence |
|---|---|---|
| 1 | Suitable documents, reference answers and testing arrangements | Linked PO and supplier-document versions; validated transcription, matches, discrepancy labels and accepted business actions; workable exports and buyer testing. Start with PC1 to PC4, then the G6 checks. Failed minimum inputs reopen the gates. |
| 2 | Arno's acceptance | To be settled with Arno before his leave at the end of October, by a short written confirmation or a session: what a check compares, where the trusted price comes from, which support he accepts (the specific reading, matching and flagging aid, with source visibility, correction and manual fallback), and who takes part in the evaluation and when. |
| 3 | Zhongxin's review | Agree scope, criteria, anchors, scores, weights and weighting procedure (option A or B); rescore the agreed boundary before final selection. |
| 4 | Johan's confirmation of business value | Confirm the company effect with examples or figures. Permission to use data is a separate decision; the score-5 anchor requires this confirmation. |

**Business relevance.** Dennis reported on 15 September that outdated purchase prices could lead Sales to sell at a loss, and the buyers lack time for regular price updates; this supports the business-value score of 3, not more. Detecting a discrepancy does not by itself maintain the price master or prevent a loss: the relevant owner must judge the applicable price and carry the outcome into the update process, and any extra review effort by Dennis must be measured separately ([role synthesis](https://github.com/Luuuuuig/Bachelor-End-Project/blob/main/docs/process/Order_Confirmation_Checking_Design_Draft_2026-10-09.md#2-denniss-existing-role-and-relevant-handoffs)).

**Next research work.** SQ2 defines permitted inputs, control points, applicable references, buyer review and fallback. SQ5 requires a manual comparator against assisted work on agreed comparable cases, with workload measures, quality guardrails, reference ownership and order effects fixed before the evaluation; the historical time per line (0.87 minutes pooled, mean rate per check 1.24 with a 95% CI of 0.70 to 1.78) is too imprecise to serve as that baseline. No additional broad observation round is proposed: the targeted evidence above is what could change the boundary, the feasibility or the ranking.

**Fallbacks.** Article/supplier data (56.8) requires its own supplier documents, field references (AR1, AR2), acceptance, export route and evaluation; request retrieval (50.8, or 57.8 on the whole Specification stage) requires access to the drawing archive, retrieval references (RI2, RI3) and evidence of how much of the stage the output would support. Their scores do not close those gates.

<a id="gate-evidence"></a>
## Appendix A. Detailed feasibility evidence

These are the original six gate verdicts, reasons and check IDs used by the calculation in Section 2. They describe evidence and unresolved checks, not a completed test-account inspection. A Redesign entry is conditional on the narrowed output described in the main text.

In [17]:
for candidate_name in gates["candidate"].drop_duplicates():
    display(pd.DataFrame({"Candidate":[candidate_name]}).style.hide(axis="index"))
    detail = gates.loc[gates["candidate"].eq(candidate_name), ["gate","verdict","reason","check"]].set_index("gate")
    display(detail.rename(columns={"verdict":"Verdict", "reason":"Evidence / boundary", "check":"Check ID"}).style.set_properties(**{"text-align":"left", "white-space":"normal"}))

Candidate
Order and confirmation checking


,Verdict,Evidence / boundary,Check ID
gate,,,
1,Pass,"Johan's permission covers using the test-account data and processing its documents with an external AI service (checks G1–G2, confirmed by Yijie on 7 October). What must be anonymised in the thesis (G3) sets how results are reported, not whether the data may be used.",
2,Provisional pass,"PO lines are standard Exact records and the observations provide examples. Whether matching confirmations/quotations exist and represent the target work remains unverified. Inspect their period and current-practice differences. If unsuitable, demonstrate a permitted representative alternative before final selection; the provisional scores do not close this gate.","PC1–PC3, G4, G5"
3,Provisional pass,"The proposed reference is the approved price per line. The buyer must verify its correctness, version and applicability to the item, quantity and order date. Availability and traceability are not yet established. Demonstrate a trustworthy source before final selection. Constructing test workloads does not supply their correct answers. For delivery date, quantity and line presence, a versioned PO is a comparison input, not automatic proof of the accepted business answer. Validate document fields, discrepancy labels and accepted actions separately.","PC4, PC6"
4,Provisional pass,The function only flags differences; the buyer judges each one and nothing changes in Exact without the buyer. On 8 October Yijie reported objection to automatic PO creation and email sending. This does not establish acceptance or rejection of flagging. Arno's acceptance of the specific support remains open.,"Arno follow-up guide, part D (rows 5–6)"
5,Provisional pass,"A prototype on exported PO lines and supplier documents could fit the BEP without Exact integration. This depends on successful exports, matched documents and buyer time for manual comparison; these checks remain open.","G6, PC1, PC7"
6,Provisional pass,"Notes describe paper, PDF, website and email prices with differing item descriptions. This may justify AI-assisted reading/matching. Inspect actual samples and compare the proposed role with structured data, OCR, search and deterministic rules. The Tater et al. precedent still requires full-text review and does not establish added value here.",PC2


Candidate
Article / supplier data


,Verdict,Evidence / boundary,Check ID
gate,,,
1,Pass,"Johan's permission covers using the test-account data and processing its documents with an external AI service (checks G1–G2, confirmed by Yijie on 7 October).",
2,Provisional pass,Article records exist in Exact; whether the supplier documents used to create them are linked is not yet known.,AR2
3,Provisional pass,"Existing article records are candidate references for extracted fields, subject to checking against source documents and buyer-approved values. Do not assume historical master data are correct.",AR1
4,Pass,The buyer checks the fields before creating the article.,
5,Provisional pass,A bounded prototype could use exported article records and supplier documents if exports and matching inputs are available; that availability remains to be checked.,"G6, AR2"
6,Provisional pass,"Different supplier-document layouts may justify AI-assisted field extraction. Inspect actual samples and compare with simpler extraction/rules; the cited benchmark is a research lead, not demonstrated performance in this setting.",


Candidate
Maximalisatie


...

,Verdict,Evidence / boundary,Check ID
gate,,,
1,Provisional pass,"Johan's permission covers using the test-account data and processing its documents with an external AI service (checks G1–G2, confirmed by Yijie on 7 October). Access to the drawing archive is not yet known.",RI2
2,Redesign,"Requests arrive by email, at the desk or by phone and are probably not stored in Exact. Narrowed output: for the item being ordered, the old POs and the drawings found in Exact or the drawing archive, for the buyer to use.","RI1, RI2"
3,Redesign,"No validated list of required drawings or complete-request definition is available for completeness assessment, so the proposed redesign excludes it. Previously used POs/drawings are candidate retrieval references; the buyer must verify their relevance, revision and completeness before evaluation.",RI3
4,Pass,Retrieval and flags only; the buyer decides.,
5,Provisional pass,Depends on access to where drawings are kept.,RI2
6,Provisional pass,Free-text item descriptions may justify AI-assisted retrieval of POs/drawings. Inspect actual examples and establish whether it adds useful support beyond ordinary search and rules.,


Candidate
Standard / review / manual


,Verdict,Evidence / boundary,Check ID
gate,,,
1,Pass,"Johan's permission covers using the test-account data and processing its documents with an external AI service (checks G1–G2, confirmed by Yijie on 7 October).",
2,Provisional pass,"PO records exist, but nothing marks a case as standard, review or manual.",SR1
3,Redesign,No reference standard exists without the buyer labelling past cases.,SR1
4,Redesign,The automatic creation-and-sending arrangement conflicts with the buyers' objection reported on 8 October. A bounded review aid requires its own acceptance check.,"Arno follow-up guide, part D"
5,Reject,"As a stand-alone candidate, an end-to-end routing redesign with Exact integration does not fit the BEP. It stays the design frame for the chosen task: cases without a flag are standard, flagged cases go to review.",
6,Provisional pass,Routing can largely be rules; AI fits only specific steps.,


Candidate
PO supplier communication


,Verdict,Evidence / boundary,Check ID
gate,,,
1,Pass,"Johan's permission covers using the test-account data and processing its documents with an external AI service (checks G1–G2, confirmed by Yijie on 7 October).",
2,Provisional pass,Sent PO emails are probably retrievable.,CM1
3,Provisional pass,"A buyer-verified recipient, attachment and message could provide a reference. A historically sent email alone does not establish correctness; validate reference examples if this supporting opportunity is pursued.",
4,Provisional pass,On 8 October Yijie reported opposition to automatic PO creation and email sending as one arrangement. Acceptance of a separate draft or send function still needs its own check; do not assume consent to automatic sending.,"Arno follow-up guide, part D (row 11)"
5,Pass,Small and buildable within the BEP.,
6,Reject,Forwarding a generated PO with a standard message is a deterministic step; ordinary automation is enough. Kept as a quick-win recommendation.,


<a id="score-evidence"></a>
## Appendix B. Score evidence, arithmetic and boundary notes

The evidence string behind every score and range, as re-examined on 12 October, then the worked arithmetic of the leader's total. The scoring evidence is Yijie's provisional judgement, drafted with AI assistance and reviewed by Yijie; no independent scoring set has been supplied.

In [18]:
for candidate_name in scores["candidate"].drop_duplicates():
    display(pd.DataFrame({"Candidate":[candidate_name]}).style.hide(axis="index"))
    detail = scores.loc[scores["candidate"].eq(candidate_name), ["criterion","score","low","high","evidence"]].set_index("criterion")
    detail.index = weights.loc[detail.index, "label"]
    display(detail.rename(columns={"score":"Score", "low":"Low", "high":"High", "evidence":"Recorded evidence"}).style.set_properties(**{"text-align":"left", "white-space":"normal"}))

Candidate
Order and confirmation checking


,Score,Low,High,Recorded evidence
label,,,,
Observed workload contribution,5,4,5,"All CHECK rows: 91 of 560 minutes (16.2%) in 19 case-days on all five days; 57 minutes (62.6%) on 18 September and 8.3% of the improvement minutes without that day; recorded difficulties on 31 Aug, 1 and 18 Sep (missing delivery date, blocked check, a mismatch difficult to explain, a large price difference) [Arno analysis 5.1, 5.2]. The notes name a price or delivery date for 7 rows / 33 minutes and an unspecified mismatch for 1 row / 20 minutes; the 7 October observer clarification covers the other 38 minutes but is not a recorded field. Low end: the dependence on 18 September."
Business value beyond counted workload,3,3,4,"Dennis reported Sales losses from outdated purchase prices (15 September); buyers lack time for updates (2 October), repeated at the 8 October meeting. Qualitative business relevance is supported. Financial impact and achievable benefit remain unquantified; Johan's confirmation with examples or figures is still to come."
AI–task fit,4,3,4,"Reading prices and lines from paper, PDF, website and e-mail documents and matching them to PO lines is the core of the task (8 Sep row 21 compares a price against the quotation and the website); precedent Tater et al. (2022) [AI table rows 5–7]. Low end: if documents are structured, only matching needs AI."
Data readiness above the gate minimum,3,2,4,"PO lines are standard Exact data; confirmations and approved prices are expected but not yet seen in the test account [checks PC1–PC4]. If they are missing, gates 2–3 reopen; this score does not absorb that."
Repetition / process standardisation,3,3,4,"Every day, 19 case-days; the task is stable (compare price and date per line) but inputs vary in format and size, 1 to 29 lines in the usable checks [Arno analysis 5.3]."
Prototype / technical feasibility,3,3,4,"Proposed route: exports and documents with manual steps, without integration; not yet demonstrated [check G6]."
Evaluation feasibility,4,3,5,"Proposed reference per line (a buyer-validated approved price) and Zhongxin's fallback of test workloads from past orders; the observed time per line is imprecise (0.87 minutes per line pooled, 0.90 on 18 September against 0.83 on the other days; mean rate per check 1.24, 95% CI 0.70 to 1.78), so the time baseline is weak [Arno analysis 5.3]."


Candidate
Article / supplier data


,Score,Low,High,Recorded evidence
label,,,,
Observed workload contribution,3,3,3,"43 of 560 minutes (7.7%) in 13 timed rows (17 ledger rows) on four days, 6 case-days; the 31 August article-master work could not be finished because required data were missing and continued on 1 September (row 33) [Arno analysis part 8 ledger]."
Business value beyond counted workload,1,1,2,No effect named beyond buyer time.
AI–task fit,4,3,4,"Extracting item fields from supplier documents in different layouts is the core of the task; precedent Šimsa et al. (2023), a benchmark [AI table row 4]."
Data readiness above the gate minimum,3,2,4,"Article records exist; whether their source documents are linked is not yet known [checks AR1, AR2]."
Repetition / process standardisation,2,2,3,"Four of five days, 6 case-days; item data differ per article."
Prototype / technical feasibility,3,2,4,"Can run on exported article records and supplier documents [checks G6, AR2]."
Evaluation feasibility,3,2,4,Existing article records can be the reference; few cases per week [check AR3].


Candidate
Maximalisatie


,Score,Low,High,Recorded evidence
label,,,,
Observed workload contribution,3,3,3,"73 of 560 minutes (13.0%) in 29 timed rows on all five days, 26 case-days [Arno analysis part 8 ledger]; 28 of those minutes are rows where MAX is named within a PO interval and 45 are rows where lines were added without a recorded search (5.0% and 8.0% of 560). The list supports finding demand to combine; the decision time itself cannot be separated."
Business value beyond counted workload,3,2,3,"One effect named: each PO costs time and money (Dennis via Yijie, 2 October); example of 4 lines added to a 12-line PO (31 Aug row 52, OBS-15)."
AI–task fit,2,1,3,"The narrowed version is close to a rule (same supplier, open PO); AI only for matching demand descriptions [AI table row 8]."
Data readiness above the gate minimum,2,1,3,Past advice and stock states are unlikely to be stored; open POs probably are [checks MX1–MX3].
Repetition / process standardisation,3,3,3,"Every day, 26 case-days in the ledger; inputs vary with stock, urgency and order value [Arno analysis part 8]."
Prototype / technical feasibility,2,1,3,"Needs open-PO and demand data, possibly Orbis [check MX4]."
Evaluation feasibility,3,2,3,The narrowed list can be checked against the lines later added to the same open PO [check MX3]; how many such cases the account holds is not yet known.


Candidate
Request intake & validation


,Score,Low,High,Recorded evidence
label,,,,
Observed workload contribution,3,3,5,"Raw definition: the Specification stage, 121 of 560 minutes (21.6%) in 33 records, 23 case-days, all five days; 108 minutes are CLAR rows with a note [Arno analysis part 8]. The gated scenario is narrowed to finding old POs and drawings, and the raw codes do not isolate that search: the six Specification rows whose note names a drawing, an old-PO lookup or an item history hold 36 minutes (6.4%, the score-3 band) on three days. Scored on the narrowed scenario; the high of 5 is the whole stage (over 15%), which would only count if the retrieval output supported most of it."
Business value beyond counted workload,2,1,3,"Incomplete drawing sets from Engineering delay orders (18 Sep OBS-10: drawings looked for, asked for and checked, rows 27, 28 and 41); no business effect beyond buyer time is named."
AI–task fit,3,2,4,"Finding the right old PO or drawing from free-text item descriptions needs AI for the search; weak precedent (Lombardi et al., 2025, other drawing domain) [AI table row 1]."
Data readiness above the gate minimum,2,1,3,"Old POs are in Exact; where drawings are kept and how they link to items is unknown [checks RI2, RI3]."
Repetition / process standardisation,2,1,3,"The retrieval-like notes fall on three of five days (5 case-days) with varying inputs: between the 1 and 3 anchors. The whole Specification stage occurs every day (23 case-days) with varying inputs, which sets the high end."
Prototype / technical feasibility,2,1,3,Depends on access to the drawing archive and on searching Exact's item history [check RI2].
Evaluation feasibility,3,2,3,"The old PO or drawing that was actually used is the reference; few cases were observed (six rows with a retrieval-like note in 5 case-days: one old-PO lookup, four rows about drawings and one item history)."


### Scores and ranges before and after the 12 October rebuild

Every score and range of the 7 October scenario next to the 12 October value, with the figure that moved and the reason. Scores whose anchor does not depend on an observed figure are unchanged by construction; the others are unchanged when the raw figure sits in the same anchor band as the earlier figure.

In [19]:
# Every score before (7 October scenario, from the archived version) and after (12 October), with the reason
previous_scores = pd.DataFrame(
    {"workload": [5, 3, 3, 3], "business_value": [3, 1, 3, 2], "ai_fit": [4, 4, 2, 3], "data_readiness": [3, 3, 2, 2],
     "repetition": [3, 2, 3, 2], "prototype": [3, 3, 2, 2], "evaluation": [4, 3, 3, 3]},
    index=["Order and confirmation checking", "Article / supplier data", "Maximalisatie", "Request intake & validation"])
previous_ranges = {("Request intake & validation", "workload"): (3, 3), ("Request intake & validation", "repetition"): (1, 2)}
reasons = {
    ("Order and confirmation checking", "workload"): "98 min / 17.5% (labels) -> 91 min / 16.2% (CHECK code): still over 15% overall, recorded on all five days, with recorded difficulties on three days",
    ("Order and confirmation checking", "repetition"): "19 case-days on every day, 1 to 29 lines: unchanged figures",
    ("Order and confirmation checking", "evaluation"): "per-line rate now 0.87 pooled; mean rate per check 1.24 with 95% CI 0.70-1.78 (Arno analysis 5.3): still an imprecise baseline",
    ("Article / supplier data", "workload"): "39 min / 7.0%, 5 case-days -> 43 min / 7.7%, 6 case-days (ledger): same 5-15% band",
    ("Article / supplier data", "repetition"): "four of five days before and after",
    ("Maximalisatie", "workload"): "43 min / 7.7% -> 73 min / 13.0% (ledger; 28 min with MAX named): same 5-15% band",
    ("Maximalisatie", "repetition"): "16 -> 26 case-days, every day before and after",
    ("Request intake & validation", "workload"): "45 min / 8.0% (retrieval labels) -> 36 min / 6.4% (retrieval-like notes): same 5-15% band, score 3; the raw definition is the whole Specification stage (121 min / 21.6%), so the high widens to 5",
    ("Request intake & validation", "repetition"): "3 of 5 days (retrieval labels) -> 3 of 5 days (retrieval-like notes), score 2; the whole Specification stage occurs every day (23 case-days), so the high widens to 3",
    ("Request intake & validation", "evaluation"): "reference unchanged; the case count now comes from the notes (6 rows, 5 case-days)",
}
rows = []
for _, row in scored.iterrows():
    key = (row["candidate"], row["criterion"])
    before = previous_scores.loc[key]
    low_before, high_before = previous_ranges.get(key, (row["low"], row["high"]))
    changed = before != row["score"] or (low_before, high_before) != (row["low"], row["high"])
    rows.append([row["candidate"], criterion_short[row["criterion"]], f"{before} ({low_before}-{high_before})",
                 f"{row['score']} ({row['low']}-{row['high']})", "changed" if changed else "unchanged",
                 reasons.get(key, "anchor does not depend on an observed figure")])
score_changes = pd.DataFrame(rows, columns=["candidate", "criterion", "7 Oct score (low-high)", "12 Oct score (low-high)", "status", "reason"])
print("Scores changed:", int((score_changes["7 Oct score (low-high)"].str[0] != score_changes["12 Oct score (low-high)"].str[0]).sum()),
      "of", len(score_changes), "- ranges changed:", int(score_changes["status"].eq("changed").sum()))
with pd.option_context("display.max_colwidth", None):
    display(score_changes.set_index(["candidate", "criterion"]).style.set_properties(**{"text-align": "left", "white-space": "normal"}))

Scores changed: 0 of 28 - ranges changed: 2


### Worked arithmetic

Each contribution is weight × score ÷ 5 × 100. This calculation uses the same matrix and full-precision weights as the ranking.

In [20]:
# Worked example: the points order and confirmation checking gets on each criterion
example = pd.DataFrame({
    "weight": weights["weight"],
    "score": score_table.loc["Order and confirmation checking"],
})
example["points"] = example["weight"] * example["score"] / 5 * 100
print("Total:", round(example["points"].sum(), 1))
example.round(3)

Total: 76.2


,weight,score,points
criterion,,,
workload,0.270,5,27.027
business_value,0.108,3,6.486
ai_fit,0.135,4,10.811
data_readiness,0.162,3,9.730
repetition,0.081,3,4.865
prototype,0.108,3,6.486
evaluation,0.135,4,10.811


### Boundary notes

- The score-5 anchor for business value requires the company supervisor's confirmation; an earlier proposal to accept a corroborated buyer account instead is not adopted.
- Checking is the CHECK code alone; no note-based rows are added and no row is split across candidates. The price mismatch of 18 September row 3 (a CLAR row in Specification: a request for information, not a purchase) is not counted; the large price difference of 18 September row 73 is. The Exact field behind the service-order mismatch remains unknown.
- The stage split of the checks (37 minutes Ordering, 54 Monitoring) is read from the agreed stage column; its provenance is in the [stage review](https://github.com/Luuuuuig/Bachelor-End-Project/blob/main/docs/measurement/Stage_Review_2026-10-12.md).
- Timed maximalisatie rows are all PO rows and are counted under both candidates; the ledger also holds two untimed DEC tallies and one request. Candidate minutes are therefore never summed.
- The 7 October gate verdicts, scores and weights are unchanged; the two widened request-intake ranges and their reasons are stated in Section 4.

<a id="sensitivity-detail"></a>
## Appendix C. Detailed perturbations and random-draw illustration

These outputs expose the calculations summarized in Section 5. The one-score loop contains clipped changes at the ends of the 1–5 scale, matching the original calculation. The combined-change search orders the available weighted one-point changes from largest to smallest to identify the fewest required to overturn the gap.

In [21]:
display(weight_checks.rename(columns={"lead_over_next":"lead_over_next_points"}).round(1))
display(score_checks.rename(columns={"lead_over_next":"lead_over_next_points"}).round(1))
with pd.option_context("display.max_colwidth", None):
    display(combined.round(1))
display(pd.concat([totals["total"].rename("Retained scenario"), equal_totals.rename("Equal weights"),
                   checking_workload_3_totals.rename("Checking workload = 3"), broad_proxy_totals.rename("Request intake on whole stage"),
                   stress_totals.rename("Joint stress")], axis=1).round(1))

,criterion,weight_change,leader,lead_over_next_points
0,workload,-20%,Order and confirmation checking,18.3
1,workload,+20%,Order and confirmation checking,20.5
2,business_value,-20%,Order and confirmation checking,19.0
3,business_value,+20%,Order and confirmation checking,19.9
4,ai_fit,-20%,Order and confirmation checking,20.0
5,ai_fit,+20%,Order and confirmation checking,18.9
6,data_readiness,-20%,Order and confirmation checking,20.1
7,data_readiness,+20%,Order and confirmation checking,18.8
8,repetition,-20%,Order and confirmation checking,19.5
9,repetition,+20%,Order and confirmation checking,19.5


,candidate,criterion,change,leader,lead_over_next_points
0,Article / supplier data,workload,-1,Order and confirmation checking,24.3
1,Article / supplier data,workload,1,Order and confirmation checking,14.1
2,Article / supplier data,business_value,-1,Order and confirmation checking,19.5
3,Article / supplier data,business_value,1,Order and confirmation checking,17.3
4,Article / supplier data,ai_fit,-1,Order and confirmation checking,22.2
5,Article / supplier data,ai_fit,1,Order and confirmation checking,16.8
6,Article / supplier data,data_readiness,-1,Order and confirmation checking,22.7
7,Article / supplier data,data_readiness,1,Order and confirmation checking,16.2
8,Article / supplier data,repetition,-1,Order and confirmation checking,21.1
9,Article / supplier data,repetition,1,Order and confirmation checking,17.8


,challenger,gap_points,changes_available,fewest_changes_needed,those_changes
0,Article / supplier data,19.5,9,6.0,Order and confirmation checking: workload -1; Order and confirmation checking: data_readiness -1; Article / supplier data: data_readiness +1; Order and confirmation checking: ai_fit -1; Order and confirmation checking: evaluation -1; Article / supplier data: evaluation +1
1,Maximalisatie,24.3,7,NaN,not possible within the ranges
2,Request intake & validation,25.4,11,7.0,Order and confirmation checking: workload -1; Request intake & validation: workload +1; Request intake & validation: workload +1; Order and confirmation checking: data_readiness -1; Request intake & validation: data_readiness +1; Order and confirmation checking: ai_fit -1; Request intake & validation: ai_fit +1


,Retained scenario,Equal weights,Checking workload = 3,Request intake on whole stage,Joint stress
candidate,,,,,
Order and confirmation checking,76.2,71.4,65.4,76.2,62.2
Article / supplier data,56.8,54.3,56.8,56.8,68.6
Maximalisatie,51.9,51.4,51.9,51.9,60.0
Request intake & validation,50.8,48.6,50.8,57.8,73.5


### Random draws

Each integer score is drawn independently and uniformly inside its recorded low/high range, with the original fixed seed and 10,000 draws. Integer swing totals identify ties exactly. These assumptions are illustrative, not an empirical model of uncertainty; strict-win shares are not probabilities that a candidate will succeed or truly rank first.

In [22]:
# 7. Random draws: each score drawn independently and uniformly from its low-high range
rng = np.random.default_rng(20261007)
draws = 10000
wins = pd.Series(0, index=score_table.index)
tied_draws = 0
for _ in range(draws):
    s = pd.DataFrame(rng.integers(low.to_numpy(), high.to_numpy() + 1), index=low.index, columns=low.columns)
    # Whole-number swing totals keep the ranking exact, so ties are found reliably
    draw_totals = s @ weights["swing"]
    leaders = draw_totals.index[draw_totals.eq(draw_totals.max())]
    if len(leaders) == 1:
        wins[leaders[0]] += 1
    else:
        tied_draws += 1

print("Strict wins by a candidate other than the main leader:", wins.sum() - wins[leader], "of", draws)
print("Draws tied for first place:", tied_draws, "of", draws)
pd.DataFrame({"strict_wins": wins, "share_of_all_draws_strictly_won_pct": wins / draws * 100}).sort_values("strict_wins", ascending=False).round(2)

Strict wins by a candidate other than the main leader: 53 of 10000
Draws tied for first place: 14 of 10000


,strict_wins,share_of_all_draws_strictly_won_pct
candidate,,
Order and confirmation checking,9933,99.33
Request intake & validation,35,0.35
Article / supplier data,18,0.18
Maximalisatie,0,0.00


<a id="sources"></a>
## Appendix D. Sources and reproducibility

Both notebooks read the same five observation CSVs (as committed on 12 October) and apply the same preparation; the candidate definitions and the two note-based ledgers are those of part 8 of the Arno analysis. The score matrix, gates and swings are explicit judgement inputs; every table, total and sensitivity derives from them, and the candidate workload table derives from the source observations.

- [Plan of Work, 22 September 2026](https://github.com/Luuuuuig/Bachelor-End-Project/blob/main/docs/proposal/Plan_of_Work_Final_2026-09-22.pdf): the five subquestions.
- [Arno analysis](https://github.com/Luuuuuig/Bachelor-End-Project/blob/main/Arno_Analysis_2026_10_02.ipynb): workload evidence and limits (parts 3 and 4), the CHECK evidence (part 5), the candidate table and ledgers (part 8).
- [Methodology, section 5.3](https://github.com/Luuuuuig/Bachelor-End-Project/blob/main/docs/methodology/Phase_1_Current_Methodology.md#53-formal-two-stage-selection-rule): gates, scoring and responsibilities; [workload definition](https://github.com/Luuuuuig/Bachelor-End-Project/blob/main/docs/methodology/Workload_Definition.md).
- [Swing record](https://github.com/Luuuuuig/Bachelor-End-Project/blob/main/docs/analyze/Swing_Weights_Yijie_2026-10-07.md), [Stage A worksheet](https://github.com/Luuuuuig/Bachelor-End-Project/blob/main/docs/analyze/Stage_A_Test_Account_Check_2026-10-07.xlsx), [stage review of 12 October](https://github.com/Luuuuuig/Bachelor-End-Project/blob/main/docs/measurement/Stage_Review_2026-10-12.md) and the [archive note](https://github.com/Luuuuuig/Bachelor-End-Project/blob/main/docs/archive/kinds_of_work/README.md) on the 7 to 9 October candidate matrix and codebook.

Belton, V., & Stewart, T. J. (2002). *Multiple criteria decision analysis: An integrated approach*. Kluwer Academic Publishers. https://doi.org/10.1007/978-1-4615-1495-4

Department for Communities and Local Government. (2009). *Multi-criteria analysis: A manual*. https://www.gov.uk/government/publications/multi-criteria-analysis-manual-for-making-government-policy

Goodhue, D. L., & Thompson, R. L. (1995). Task-technology fit and individual performance. *MIS Quarterly, 19*(2), 213–236. https://doi.org/10.2307/249689

Parasuraman, R., Sheridan, T. B., & Wickens, C. D. (2000). A model for types and levels of human interaction with automation. *IEEE Transactions on Systems, Man, and Cybernetics – Part A, 30*(3), 286–297. https://doi.org/10.1109/3468.844354

Peffers, K., Tuunanen, T., Rothenberger, M. A., & Chatterjee, S. (2007). A design science research methodology for information systems research. *Journal of Management Information Systems, 24*(3), 45–77.

Tater, T., et al. (2022). AI driven accounts payable transformation. *Proceedings of the AAAI Conference on Artificial Intelligence, 36*(11), 12405–12413. https://doi.org/10.1609/aaai.v36i11.21506

Triantaphyllou, E., & Sánchez, A. (1997). A sensitivity analysis approach for some deterministic multi-criteria decision-making methods. *Decision Sciences, 28*(1), 151–194. https://doi.org/10.1111/j.1540-5915.1997.tb01306.x